# Real-World Agent Use Cases

AI agents are everywhere in product marketing — but not every problem needs one. This notebook surveys **production-shaped use cases** where agents genuinely help, shows **working miniature implementations** for each, and gives you a **decision framework** for when *not* to build an agent.

We cover six common patterns:

1. **RAG assistants** — answer from private knowledge with citations.
2. **Coding agents** — read, edit, and verify code in a repository.
3. **Customer support agents** — triage tickets, look up policies, draft replies.
4. **Data analyst agents** — turn questions into SQL and summarize results.
5. **Internal documentation assistants** — help employees navigate company wikis.
6. **Research crews** — multiple specialized roles exploring a topic.

Each pattern includes runnable Python code using in-memory data — no external services, no prior notebooks, and no CLI required. The goal is to understand **what real agents do in the wild** so you can match architecture to problem, not hype.

In [ ]:
"""
Shared imports and sample data for all use-case demos in this notebook.
"""

import json
import os
import re
import sqlite3
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Any, Callable

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-proj-placeholder-replace-with-your-real-key")
os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)


def keyword_search(docs: list[dict[str, str]], query: str, top_k: int = 3) -> list[dict[str, str]]:
    """Simple keyword retriever used by RAG and docs assistants."""
    terms = [t for t in re.split(r"\W+", query.lower()) if len(t) > 2]
    if not terms:
        return []
    scored = []
    for doc in docs:
        haystack = f"{doc.get('title', '')} {doc.get('text', '')}".lower()
        score = sum(1 for term in terms if term in haystack)
        if score:
            scored.append((score, doc))
    scored.sort(key=lambda x: (-x[0], x[1].get("id", "")))
    return [d for _, d in scored[:top_k]]


print("Shared utilities loaded.")

---

## 1. The Use Case Landscape

Most deployed agents fall into a few clusters:

```
                    REAL-WORLD AGENT USE CASES
    ┌────────────┬────────────┬────────────┬────────────┐
    │    RAG     │   Coding   │  Support   │  Analyst   │
    │ assistant  │   agent    │    bot     │   agent    │
    └─────┬──────┴─────┬──────┴─────┬──────┴─────┬──────┘
          │            │            │            │
          └────────────┴────────────┴────────────┘
                         │
              internal docs assistants
              research crews (multi-role)
```

| Cluster | Core job | Typical tools | Agent shape |
|---------|----------|---------------|-------------|
| **Knowledge** | Find and synthesize information | Search, retrieve, cite | Single agent + search tool |
| **Action** | Change systems (code, tickets, orders) | File edit, API call, SQL | Single agent + many tools |
| **Collaboration** | Explore open-ended topics | Web search, doc analysis, writing | Multi-agent crew |

The sections below implement one miniature version of each cluster so you can see the control flow, not just read bullet points.

---

## 2. RAG Assistants — Retrieve, Reason, Respond

A **RAG (Retrieval-Augmented Generation) assistant** answers questions grounded in private documents. The agent loop typically looks like:

1. User asks a question.
2. Agent calls a **search tool** over a vector DB or keyword index.
3. Agent reads retrieved chunks and composes an answer **with citations**.

| Aspect | Typical design |
|--------|----------------|
| **Goal** | Accurate answers from company knowledge |
| **Tools** | Vector/keyword search, optional reranker |
| **Agent?** | Often a single-agent ReAct loop over retrieval |
| **Main risk** | Stale index, hallucination without retrieval |

Production tip: force a search step before answering factual questions — do not let the model skip retrieval.

In [ ]:
# Sample private knowledge base (product documentation)
PRODUCT_DOCS: list[dict[str, str]] = [
    {
        "id": "doc-billing",
        "title": "Billing and refunds",
        "text": "Refunds are available within 30 days of purchase. Pro plans are billed annually. Contact billing@acme.com for invoice copies.",
    },
    {
        "id": "doc-auth",
        "title": "Authentication",
        "text": "Users sign in with email/password or SSO. API keys are scoped per workspace. Rotate keys from Settings > Security.",
    },
    {
        "id": "doc-limits",
        "title": "Rate limits",
        "text": "Free tier: 100 API requests per hour. Pro tier: 10,000 requests per hour. Burst limits apply for 60-second windows.",
    },
]


@dataclass
class RAGAssistant:
    """Single-agent RAG: search → compose cited answer."""

    docs: list[dict[str, str]] = field(default_factory=lambda: list(PRODUCT_DOCS))
    trace: list[str] = field(default_factory=list)

    def search(self, query: str) -> list[dict[str, str]]:
        hits = keyword_search(self.docs, query, top_k=2)
        self.trace.append(f"search(query={query!r}) → {len(hits)} hits")
        return hits

    def answer(self, question: str) -> str:
        self.trace = []
        hits = self.search(question)
        if not hits:
            return "I could not find documentation for that question."
        cites = ", ".join(f"[{h['id']}]" for h in hits)
        self.trace.append("compose answer from retrieved chunks")
        return f"Based on {cites}: {hits[0]['text']}"


rag = RAGAssistant()
q = "How do API rate limits work on the free tier?"
print(f"Q: {q}")
print(f"A: {rag.answer(q)}")
print(f"Trace: {rag.trace}")

---

## 3. Coding Agents — Read, Edit, Verify

Coding agents help developers inside IDEs and CI pipelines. They loop over:

1. **Read** relevant files.
2. **Edit** code (patch or full rewrite).
3. **Verify** with tests or linters.
4. **Repeat** until tests pass or step limit is hit.

| Capability | Tool examples | Risk |
|------------|---------------|------|
| Read | `read_file`, AST search | Low |
| Write | `apply_patch`, `write_file` | Medium — unreviewed changes |
| Verify | `run_tests`, linter | Low if sandboxed |
| Execute | Sandboxed shell | **High** — arbitrary code execution |

Production coding agents almost always run in a **sandbox**, require **diff approval**, and cap **max steps**.

In [ ]:
# Simulated repository (file path → content)
REPO: dict[str, str] = {
    "src/calculator.py": "def add(a, b):\n    return a - b  # bug: should be +\n",
    "tests/test_calculator.py": (
        "from src.calculator import add\n\n"
        "def test_add():\n"
        "    assert add(2, 3) == 5\n"
    ),
}


def read_file(path: str) -> str:
    if path not in REPO:
        raise FileNotFoundError(path)
    return REPO[path]


def apply_patch(path: str, old: str, new: str) -> dict[str, Any]:
    content = read_file(path)
    if old not in content:
        return {"success": False, "error": "old snippet not found"}
    REPO[path] = content.replace(old, new, 1)
    return {"success": True, "path": path}


def run_tests() -> dict[str, Any]:
    """Execute the calculator test logic inline (no subprocess)."""
    try:
        namespace: dict[str, Any] = {}
        exec(REPO["src/calculator.py"], namespace)
        add = namespace["add"]
        assert add(2, 3) == 5, "add(2, 3) should equal 5"
        return {"passed": True, "failures": []}
    except Exception as exc:
        return {"passed": False, "failures": [str(exc)]}


@dataclass
class CodingAgent:
    """Minimal coding agent: read → patch → test loop."""

    max_steps: int = 5
    trace: list[str] = field(default_factory=list)

    def run_fix_add_bug(self) -> dict[str, Any]:
        self.trace = []

        # Reset repo so demo works when cells are re-run in the same kernel
        REPO["src/calculator.py"] = "def add(a, b):\n    return a - b  # bug: should be +\n"

        # Step 1: read failing test
        test_src = read_file("tests/test_calculator.py")
        self.trace.append(f"read tests/test_calculator.py ({len(test_src)} chars)")

        # Step 2: read implementation
        impl = read_file("src/calculator.py")
        self.trace.append(f"read src/calculator.py — spotted subtraction bug")

        # Step 3: verify failure
        before = run_tests()
        self.trace.append(f"run_tests before patch → passed={before['passed']}")

        # Step 4: apply fix
        patch_result = apply_patch("src/calculator.py", "return a - b", "return a + b")
        self.trace.append(f"apply_patch → {patch_result}")

        # Step 5: verify success
        after = run_tests()
        self.trace.append(f"run_tests after patch → passed={after['passed']}")

        return {"tests_passed": after["passed"], "trace": self.trace}


coding_agent = CodingAgent()
result = coding_agent.run_fix_add_bug()
print("Tests passed:", result["tests_passed"])
for step in result["trace"]:
    print(f"  {step}")

---

## 4. Customer Support Agents

Support agents handle L1 tickets: classify intent, look up policy, draft a reply, and escalate when needed.

```
 ticket ──► triage agent ──► policy lookup ──► draft reply
                  │                    │
                  └── escalate human ──┘
```

| Pattern | When to use | Example |
|---------|-------------|---------|
| **Retrieve + respond** | FAQ, policy questions | Refund window lookup |
| **Tool-backed** | Needs live data | Order status, account tier |
| **Human handoff** | Legal risk, angry customer | Threats, chargebacks |

Guardrails matter most here: never fabricate policy, cap refund authority, and route high-risk intents to humans.

In [ ]:
SUPPORT_KB: list[dict[str, str]] = [
    {"id": "pol-refund", "title": "Refund policy", "text": "Full refunds within 30 days. Partial refunds require manager approval."},
    {"id": "pol-shipping", "title": "Shipping delays", "text": "Standard shipping is 5-7 business days. Expedited options available at checkout."},
]

ORDERS: dict[str, dict[str, Any]] = {
    "9912": {"status": "delivered", "days_since_purchase": 45, "amount": 89.99},
    "9913": {"status": "shipped", "days_since_purchase": 3, "amount": 24.50},
}


def lookup_order(order_id: str) -> dict[str, Any]:
    order = ORDERS.get(order_id)
    if not order:
        return {"found": False}
    return {"found": True, "order_id": order_id, **order}


def triage_message(text: str) -> str:
    t = text.lower()
    if any(w in t for w in ("lawsuit", "lawyer", "fraud")):
        return "escalate"
    if "refund" in t:
        return "refund"
    if "shipping" in t or "delivery" in t:
        return "shipping"
    return "general"


@dataclass
class SupportAgent:
    trace: list[str] = field(default_factory=list)

    def handle(self, customer_message: str) -> dict[str, Any]:
        self.trace = []
        intent = triage_message(customer_message)
        self.trace.append(f"triage → intent={intent}")

        if intent == "escalate":
            return {
                "route": "human",
                "reply": "Connecting you with a senior support specialist.",
                "trace": self.trace,
            }

        if intent == "refund":
            order_id = re.search(r"#(\d+)", customer_message)
            if order_id:
                order = lookup_order(order_id.group(1))
                self.trace.append(f"lookup_order → {order}")
                if order.get("found") and order["days_since_purchase"] > 30:
                    policy = keyword_search(SUPPORT_KB, "refund")[0]
                    return {
                        "route": "auto",
                        "reply": (
                            f"Order {order['order_id']} is outside the 30-day window [{policy['id']}]. "
                            "I can request a manager review if you'd like."
                        ),
                        "trace": self.trace,
                    }

        hits = keyword_search(SUPPORT_KB, customer_message, top_k=1)
        self.trace.append(f"search_kb → {hits[0]['id'] if hits else 'none'}")
        if hits:
            return {
                "route": "auto",
                "reply": f"[{hits[0]['id']}] {hits[0]['text']}",
                "trace": self.trace,
            }
        return {"route": "human", "reply": "Let me connect you with an agent.", "trace": self.trace}


support = SupportAgent()
for msg in [
    "I want a refund on order #9912",
    "Where is my package for order #9913?",
    "I will contact my lawyer about this fraud",
]:
    out = support.handle(msg)
    print(f"Customer: {msg}")
    print(f"  Route: {out['route']} | Reply: {out['reply'][:80]}...")
    print()

---

## 5. Data Analyst Agents

Analyst agents translate natural language into **SQL or Python**, execute **read-only** queries, and summarize findings.

| Step | Action |
|------|--------|
| 1 | Parse the business question |
| 2 | Generate SQL via tool or structured output |
| 3 | Execute against a **read-only** database role |
| 4 | Summarize results and suggest a chart type |

**Critical guardrail:** block `DELETE`, `UPDATE`, `DROP`, and `INSERT` at the parser level — never rely on the model alone.

In [ ]:
# In-memory SQLite warehouse (events table)
DB = sqlite3.connect(":memory:")
DB.row_factory = sqlite3.Row
DB.executescript("""
CREATE TABLE events (
    id INTEGER PRIMARY KEY,
    user_id TEXT NOT NULL,
    event_type TEXT NOT NULL,
    created_at TEXT NOT NULL
);
INSERT INTO events (user_id, event_type, created_at) VALUES
    ('u1', 'login', '2026-07-01'),
    ('u2', 'login', '2026-07-02'),
    ('u1', 'purchase', '2026-07-03'),
    ('u3', 'login', '2026-07-08'),
    ('u2', 'purchase', '2026-07-09'),
    ('u4', 'login', '2026-07-09');
""")

BLOCKED_SQL = re.compile(r"\b(INSERT|UPDATE|DELETE|DROP|ALTER|CREATE)\b", re.IGNORECASE)


def execute_readonly_sql(sql: str) -> dict[str, Any]:
    if BLOCKED_SQL.search(sql):
        return {"success": False, "error": "Only SELECT queries are allowed."}
    if not sql.strip().upper().startswith("SELECT"):
        return {"success": False, "error": "Query must start with SELECT."}
    try:
        rows = DB.execute(sql).fetchall()
        return {"success": True, "rows": [dict(r) for r in rows], "row_count": len(rows)}
    except Exception as exc:
        return {"success": False, "error": str(exc)}


QUESTION_TO_SQL = {
    "weekly active users": "SELECT COUNT(DISTINCT user_id) AS wau FROM events WHERE created_at >= '2026-07-08'",
    "login events": "SELECT event_type, COUNT(*) AS cnt FROM events WHERE event_type = 'login' GROUP BY event_type",
    "purchases": "SELECT COUNT(*) AS purchase_count FROM events WHERE event_type = 'purchase'",
}


@dataclass
class AnalystAgent:
    trace: list[str] = field(default_factory=list)

    def generate_sql(self, question: str) -> str:
        q = question.lower()
        for key, sql in QUESTION_TO_SQL.items():
            if key in q:
                return sql
        return "SELECT event_type, COUNT(*) AS cnt FROM events GROUP BY event_type"

    def run(self, question: str) -> dict[str, Any]:
        self.trace = []
        self.trace.append(f"parse question: {question!r}")
        sql = self.generate_sql(question)
        self.trace.append(f"generate_sql: {sql}")
        result = execute_readonly_sql(sql)
        self.trace.append(f"execute → success={result.get('success')}")
        if not result["success"]:
            return {"summary": result["error"], "trace": self.trace}
        summary = f"Query returned {result['row_count']} row(s): {result['rows']}"
        self.trace.append("summarize + suggest bar chart")
        return {"sql": sql, "data": result["rows"], "summary": summary, "trace": self.trace}


analyst = AnalystAgent()
for question in ["How many weekly active users?", "Show login events", "DELETE FROM events"]:
    print(f"Q: {question}")
    if "DELETE" in question.upper():
        blocked = execute_readonly_sql("DELETE FROM events")
        print(f"  Blocked: {blocked['error']}")
    else:
        out = analyst.run(question)
        print(f"  {out['summary']}")
    print()

---

## 6. Internal Documentation Assistants

Large companies have wikis, runbooks, and architecture docs scattered across systems. An internal docs assistant helps engineers **find the right page fast** and **cite the source**.

This is structurally similar to RAG, but tuned for:

- **Technical vocabulary** (service names, internal codenames).
- **Freshness** — docs change weekly; index must update.
- **Access control** — not every employee sees every doc.

| User question | Agent path | Expected behavior |
|---------------|------------|-------------------|
| "How do I deploy service X?" | Search runbooks → cite | Step-by-step with doc ID |
| "What's our on-call rotation?" | Search ops docs | Link to schedule page |
| "Who owns the payments API?" | Search + optional ownership DB | Name + team channel |

In [ ]:
INTERNAL_WIKI: list[dict[str, str]] = [
    {
        "id": "runbook-deploy",
        "title": "Deploying the payments API",
        "text": "Merge to main → CI green → deploy via ArgoCD to staging → smoke test → promote to prod with change ticket CHG-1001.",
    },
    {
        "id": "runbook-oncall",
        "title": "On-call rotation",
        "text": "Primary on-call rotates weekly. Schedule lives in PagerDuty. Escalation path: primary → secondary → engineering manager.",
    },
    {
        "id": "owners-payments",
        "title": "Service ownership — payments",
        "text": "Payments API owned by Team Ledger. Slack: #team-ledger. Tech lead: priya@company.com.",
    },
]


@dataclass
class InternalDocsAssistant:
    docs: list[dict[str, str]] = field(default_factory=lambda: list(INTERNAL_WIKI))

    def answer(self, question: str) -> str:
        hits = keyword_search(self.docs, question, top_k=2)
        if not hits:
            return "No internal documentation found. Try the search portal or ask in #engineering."
        lines = [f"[{h['id']}] {h['title']}: {h['text']}" for h in hits]
        return "\n".join(lines)


docs_bot = InternalDocsAssistant()
queries = [
    "How do I deploy the payments API to production?",
    "Who owns the payments service?",
    "What is the on-call escalation path?",
]
for q in queries:
    print(f"Q: {q}")
    print(docs_bot.answer(q))
    print()

---

## 7. Research Crews — Multi-Agent Exploration

Open-ended research benefits from **role separation** — one model trying to search, analyze, and write in a single prompt often produces shallow results.

```
 lead researcher (planner)
      │
      ├── search specialist — finds raw sources
      ├── analyst — extracts facts and compares
      └── writer — synthesizes final report
```

Each role can use a different system prompt, tool set, and even model size. A **planner** coordinates handoffs.

In [ ]:
RESEARCH_SOURCES: list[dict[str, str]] = [
    {"id": "s1", "topic": "vector databases", "text": "Pinecone and Weaviate are managed vector DBs; pgvector adds vectors to PostgreSQL."},
    {"id": "s2", "topic": "vector databases", "text": "HNSW is a common approximate nearest neighbor index; recall vs latency tradeoff."},
    {"id": "s3", "topic": "vector databases", "text": "Embedding model choice affects retrieval quality more than index algorithm for small corpora."},
]


def search_specialist(topic: str) -> list[dict[str, str]]:
    return keyword_search(RESEARCH_SOURCES, topic, top_k=3)


def analyst_role(sources: list[dict[str, str]]) -> list[str]:
    return [f"Finding from [{s['id']}]: {s['text'][:70]}..." for s in sources]


def writer_role(findings: list[str], topic: str) -> str:
    body = " ".join(findings)
    return f"Research brief on {topic}: {body}"


@dataclass
class ResearchCrew:
    trace: list[dict[str, str]] = field(default_factory=list)

    def run(self, topic: str) -> dict[str, Any]:
        self.trace = []

        self.trace.append({"role": "planner", "action": f"Delegate research on '{topic}'"})

        sources = search_specialist(topic)
        self.trace.append({"role": "search_specialist", "action": f"Found {len(sources)} sources"})

        findings = analyst_role(sources)
        self.trace.append({"role": "analyst", "action": f"Extracted {len(findings)} findings"})

        report = writer_role(findings, topic)
        self.trace.append({"role": "writer", "action": "Synthesized final brief"})

        return {"report": report, "trace": self.trace}


crew = ResearchCrew()
output = crew.run("vector databases for RAG")
for step in output["trace"]:
    print(f"[{step['role']}] {step['action']}")
print(f"\nReport: {output['report'][:200]}...")

---

## 8. Case Study Comparison Table

Use this table when designing or pitching a new AI feature:

| Use case | Needs agent? | Typical architecture | Tool count | Max steps | Primary risk |
|----------|--------------|----------------------|------------|-----------|--------------|
| **RAG FAQ** | Often yes (light) | Single agent + search | 1–3 | 5–8 | Stale index, skipped retrieval |
| **Coding agent** | Yes | Read/write/test loop in sandbox | 5–15 | 15–30 | Arbitrary code execution |
| **Support L1** | Sometimes | Triage + KB + CRM tools | 2–5 | 5–10 | Wrong refund, fabricated policy |
| **Data analyst** | Yes | SQL tool + read-only DB | 2–4 | 5–12 | Data leak, destructive SQL |
| **Internal docs** | Yes (light) | Search + cite | 1–2 | 3–6 | Hallucinated policy |
| **Research crew** | Yes (multi) | Planner + 2–4 specialists | 3–8 per role | 10–25 total | Cost, runaway loops |
| **Static FAQ page** | **No** | CMS / search | 0 | 1 | N/A |
| **Translation** | **No** | Single LLM call | 0 | 1 | N/A |

In [ ]:
CASE_STUDIES: list[dict[str, Any]] = [
    {"use_case": "RAG FAQ", "agents": "single", "tools": 2, "max_steps": 6, "risk": "stale index"},
    {"use_case": "Coding agent", "agents": "single", "tools": 8, "max_steps": 20, "risk": "code exec"},
    {"use_case": "Support L1", "agents": "single", "tools": 4, "max_steps": 8, "risk": "wrong refund"},
    {"use_case": "Data analyst", "agents": "single", "tools": 3, "max_steps": 10, "risk": "SQL safety"},
    {"use_case": "Research crew", "agents": "multi", "tools": 6, "max_steps": 20, "risk": "cost / loops"},
]

print(f"{'Use case':<16} {'Agents':<8} {'Tools':>5} {'Steps':>6}  Risk")
print("-" * 55)
for row in CASE_STUDIES:
    print(f"{row['use_case']:<16} {row['agents']:<8} {row['tools']:>5} {row['max_steps']:>6}  {row['risk']}")

---

## 9. When NOT to Use Agents

Agents are expensive, slow, and nondeterministic compared to traditional software. Prefer simpler approaches when:

| Situation | Better alternative | Why |
|-----------|-------------------|-----|
| Single-step transformation | Prompt template or function | No loop needed |
| Deterministic ETL | Code pipeline (Airflow, dbt) | Predictable, testable |
| Strict UI workflow | Form wizard | User expects fixed steps |
| Sub-100ms latency | Cached lookup or rules engine | Agents are too slow |
| Zero-error finance | Rules engine + human approval | Model cannot guarantee correctness |
| Well-defined API mapping | Traditional integration | No reasoning required |
| Translation / summarization of given text | Single LLM call | No tools or state needed |

In [ ]:
def recommend_approach(requirements: dict[str, bool]) -> str:
    """Decision helper — returns the simplest adequate architecture."""
    if requirements.get("deterministic") and not requirements.get("ambiguous"):
        return "traditional code / workflow engine"
    if requirements.get("single_step") and not requirements.get("needs_live_data"):
        return "single LLM call or prompt template"
    if requirements.get("needs_live_data") and requirements.get("multi_step"):
        if requirements.get("parallel_specialists"):
            return "multi-agent crew"
        return "single-agent with tools (start here)"
    if requirements.get("needs_private_docs"):
        return "RAG assistant (search + answer)"
    return "re-evaluate requirements"


SCENARIOS = [
    ("Hash password with bcrypt", {"deterministic": True, "ambiguous": False, "single_step": True}),
    ("Answer billing FAQ from help center", {"needs_private_docs": True, "multi_step": False}),
    ("Fix failing unit test in repo", {"needs_live_data": True, "multi_step": True, "deterministic": False}),
    ("Compare 3 cloud vendors for ML training", {"parallel_specialists": True, "multi_step": True, "ambiguous": True}),
    ("Translate product description to French", {"single_step": True, "needs_live_data": False}),
]

for name, reqs in SCENARIOS:
    print(f"{recommend_approach(reqs):<35} ← {name}")

---

## 10. End-to-End Scenario — New Engineer Onboarding

A realistic internal request: *"I'm new — how do I deploy the payments API and who do I ask if it breaks?"*

This requires **multiple capabilities** from our demos:

1. **Internal docs search** — deployment runbook.
2. **Ownership lookup** — team channel and contact.
3. **Synthesis** — numbered checklist for the new engineer.

Below, a small orchestrator chains the docs assistant without needing a separate multi-agent framework.

In [ ]:
@dataclass
class OnboardingOrchestrator:
    """Chains internal docs lookups for a compound new-hire question."""

    docs: InternalDocsAssistant = field(default_factory=InternalDocsAssistant)
    trace: list[str] = field(default_factory=list)

    def run(self, question: str) -> str:
        self.trace = []
        steps = [
            "How do I deploy the payments API to production?",
            "Who owns the payments service?",
            "What is the on-call escalation path?",
        ]
        sections = []
        for i, sub_q in enumerate(steps, 1):
            answer = self.docs.answer(sub_q)
            self.trace.append(f"sub-query {i}: {sub_q}")
            sections.append(f"{i}. {answer}")
        return "\n\n".join(sections)


onboarding = OnboardingOrchestrator()
question = "I'm new — how do I deploy payments API and who do I ask if it breaks?"
print(f"USER: {question}\n")
print(onboarding.run(question))
print(f"\nOrchestrator trace: {onboarding.trace}")

---

## 11. Industry Patterns vs Hype

| Hype claim | Reality |
|------------|---------|
| "Agents will replace all software" | Agents **orchestrate** existing APIs and UIs — they rarely replace them |
| "Fully autonomous, no humans" | Human-in-the-loop and approval gates are normal in production |
| "One super-agent does everything" | Specialists + supervisor scale better and are easier to maintain |
| "No evaluation needed" | Every use case needs metrics — citation accuracy, tests passing, resolution rate |
| "Agents are always smarter" | Often slower and costlier than a form, cache, or script |

---

## 12. Evaluation Hooks per Use Case

Before shipping, define how you will measure success:

| Use case | Key metrics | Example failure signal |
|----------|-------------|------------------------|
| **RAG** | Citation accuracy, faithfulness | Answer cites wrong doc ID |
| **Coding** | Tests pass, diff size, review acceptance | Patch breaks unrelated tests |
| **Support** | Resolution rate, escalation rate, CSAT | Auto-refund outside policy |
| **Analyst** | Query correctness, row-level safety | `SELECT *` on PII table |
| **Internal docs** | Retrieval recall@k, time-to-answer | Hallucinated deploy steps |
| **Research crew** | Source coverage, factual consistency | Misses major competitor |

---

## 13. Common Pitfalls by Domain

| Domain | Pitfall | Mitigation |
|--------|---------|------------|
| **RAG** | Model answers without retrieving | Force search tool on first step |
| **Coding** | Unreviewed writes to production | Diff approval, branch protection |
| **Support** | Fabricated refund policy | Retrieve-only mode for policy text |
| **Analyst** | Destructive SQL | Parser blocklist + read-only DB role |
| **Internal docs** | Outdated index | Scheduled re-ingestion, version tags |
| **Research crew** | Runaway cost | Per-role step limits, token budgets |

---

## 14. Hands-On — Classify Real Requests

For each request below, identify the use case cluster and whether an agent is warranted.

In [ ]:
def classify_use_case(text: str) -> tuple[str, str]:
    t = text.lower()
    if any(w in t for w in ("pytest", "refactor", "bug", "pull request", "unit test")):
        return "Coding", "agent"
    if any(w in t for w in ("refund", "order", "shipping", "customer")):
        return "Support", "agent or RAG"
    if any(w in t for w in ("select", "sql", "weekly active", "events table", "chart")):
        return "Analyst", "agent"
    if any(w in t for w in ("deploy", "runbook", "on-call", "who owns")):
        return "Internal docs", "RAG / light agent"
    if any(w in t for w in ("research", "compare", "landscape", "vendors")):
        return "Research crew", "multi-agent"
    if any(w in t for w in ("translate", "french", "spanish")):
        return "Translation", "not an agent"
    if any(w in t for w in ("chunk", "documentation", "policy", "what does")):
        return "RAG", "RAG assistant"
    return "General", "single LLM or RAG"


CASES = [
    "What does our refund policy say for order #9912?",
    "Add a pytest for POST /orders returning 201",
    "How many weekly active users from the events table?",
    "Who owns the payments API and how do I deploy it?",
    "Research and compare three vector database vendors",
    "Translate this sentence to French",
]

print(f"{'Use case':<16} {'Recommendation':<20} Request")
print("-" * 75)
for text in CASES:
    use_case, rec = classify_use_case(text)
    print(f"{use_case:<16} {rec:<20} {text}")

---

## 15. Unified Demo — One Entry Point, Six Backends

The `UseCaseRouter` below shows how a product might route incoming requests to the right agent pattern — similar to a real platform with multiple specialized backends.

In [ ]:
@dataclass
class UseCaseRouter:
    rag: RAGAssistant = field(default_factory=RAGAssistant)
    coding: CodingAgent = field(default_factory=CodingAgent)
    support: SupportAgent = field(default_factory=SupportAgent)
    analyst: AnalystAgent = field(default_factory=AnalystAgent)
    docs: InternalDocsAssistant = field(default_factory=InternalDocsAssistant)
    crew: ResearchCrew = field(default_factory=ResearchCrew)

    def handle(self, request: str) -> dict[str, Any]:
        use_case, _ = classify_use_case(request)

        if use_case == "Coding":
            return {"use_case": use_case, "result": self.coding.run_fix_add_bug()}
        if use_case == "Support":
            return {"use_case": use_case, "result": self.support.handle(request)}
        if use_case == "Analyst":
            return {"use_case": use_case, "result": self.analyst.run(request)}
        if use_case == "Internal docs":
            return {"use_case": use_case, "result": self.docs.answer(request)}
        if use_case == "Research crew":
            return {"use_case": use_case, "result": self.crew.run(request)}
        if use_case == "RAG":
            return {"use_case": use_case, "result": self.rag.answer(request)}
        return {"use_case": use_case, "result": "Route to single LLM call — no agent loop needed."}


router = UseCaseRouter()
for req in [
    "What does our refund policy say?",
    "How many weekly active users from the events table?",
]:
    out = router.handle(req)
    print(f"[{out['use_case']}] {req}")
    result = out["result"]
    if isinstance(result, dict) and "summary" in result:
        print(f"  → {result['summary']}")
    elif isinstance(result, str):
        print(f"  → {result[:100]}...")
    else:
        print(f"  → {str(result)[:100]}...")
    print()

---

## 16. Optional — Live LLM Use Case Brainstorm

Set `USE_LIVE_LLM = True` with a valid API key to ask a model for one internal-docs agent use case.

In [ ]:
USE_LIVE_LLM = False

OFFLINE_EXAMPLE = (
    "Internal-docs agent use case: New hire asks 'How do I rotate API keys for staging?' — "
    "agent searches the security runbook, cites the key-rotation section, and links the approval form."
)

if USE_LIVE_LLM:
    try:
        from openai import OpenAI
        client = OpenAI()
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{
                "role": "user",
                "content": "Give one concrete internal-documentation agent use case in two sentences.",
            }],
            max_tokens=80,
        )
        print(resp.choices[0].message.content)
    except Exception as exc:
        print("LLM call failed:", exc)
else:
    print(OFFLINE_EXAMPLE)

---

## 17. Check Your Understanding

1. What is the typical tool loop for a **RAG assistant**?
2. Why do **coding agents** need sandboxes and diff approval?
3. When should a **support agent** escalate to a human instead of auto-replying?
4. What SQL guardrail does the analyst demo enforce, and why is it not optional?
5. How is a **research crew** different from a single agent with many tools?
6. Name two requests from section 14 that should **not** use an agent loop.
7. What metrics would you track for a RAG FAQ vs a coding agent?

---

## 18. Summary

Real-world agents cluster around **knowledge work** (RAG, internal docs), **action work** (coding, support tools, SQL), and **collaborative research** (multi-role crews). The right architecture depends on the problem — not on whether "agents" are trending.

**Key takeaways:**

- **RAG assistants** search private docs and cite sources — force retrieval before answering.
- **Coding agents** read, patch, and test in a loop — sandbox execution and cap steps.
- **Support agents** triage, look up policy and orders, and escalate high-risk intents.
- **Analyst agents** generate read-only SQL and summarize — block destructive statements in code.
- **Internal docs assistants** are RAG tuned for engineering vocabulary and ownership metadata.
- **Research crews** split search, analysis, and writing across roles.
- Many problems are better solved with **forms, caches, pipelines, or single LLM calls** — use the decision helper in section 9.
- Start **single-agent + tools**; add multi-agent only when roles or prompts conflict.

You can now evaluate any product idea: *Which use case cluster is it? Does it pass the "when not to use agents" test? What tools, steps, and metrics apply?*